# Task-4 Week-4.

**Dataset :-** 'Admission_Prediction.csv'

**Objective :- Develop integrated dashboards combining multiple interactive data visualizations, providing real-time insights into model performance, admissions KPIs, and applicant data exploration.

In [43]:
# Import Library

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [44]:
# Load Dataset

df = pd.read_csv("Admission_Prediction.csv")

print("Dataset loaded successfully!")
print("Number of rows and columns:", df.shape)

display(df.head())

print("Column names:")
print(df.columns.tolist())

print("\nMissing values:")
display(df.isnull().sum())

print("\nData types:")
display(df.dtypes)

Dataset loaded successfully!
Number of rows and columns: (1000, 17)


,Serial No.,GRE Score,TOEFL Score,University Rating,SOP,LOR,CGPA,Research,Chance of Admit,Year,Application_Date,University_Name,region,Admission Status,Duration_Months,Dropout_Event,Dropout-free survival
0,104,317,104,2,4.5,4.0,8.47,0,0.57,2022,03-01-2022,Indian Institute of Technology Bombay,West,Rejected,36,0,0.536473
1,170,311,99,2,2.5,3.0,7.98,0,0.65,2022,09-01-2022,Indian Institute of Technology Delhi,North,Rejected,18,1,0.973000
2,138,316,100,2,1.5,3.0,8.16,1,0.71,2022,12-01-2022,Indian Institute of Technology Delhi,North,Rejected,33,0,0.597989
3,147,315,105,3,2.0,2.5,8.48,0,0.75,2022,13-01-2022,Indian Institute of Technology Delhi,North,Accepted,33,0,0.597989
4,117,299,102,3,4.0,3.5,8.62,0,0.56,2022,14-01-2022,Indian Institute of Technology Delhi,North,Rejected,28,1,0.734170


Column names:
['Serial No.', 'GRE Score', 'TOEFL Score', 'University Rating', 'SOP', 'LOR', 'CGPA', 'Research', 'Chance of Admit', 'Year', 'Application_Date', 'University_Name', 'region', 'Admission Status', 'Duration_Months', 'Dropout_Event', 'Dropout-free survival']

Missing values:


Serial No.               0
GRE Score                0
TOEFL Score              0
University Rating        0
SOP                      0
LOR                      0
CGPA                     0
Research                 0
Chance of Admit          0
Year                     0
Application_Date         0
University_Name          0
region                   0
Admission Status         0
Duration_Months          0
Dropout_Event            0
Dropout-free survival    0
dtype: int64


Data types:


Serial No.                 int64
GRE Score                  int64
TOEFL Score                int64
University Rating          int64
SOP                      float64
LOR                      float64
CGPA                     float64
Research                   int64
Chance of Admit          float64
Year                       int64
Application_Date          object
University_Name           object
region                    object
Admission Status          object
Duration_Months            int64
Dropout_Event              int64
Dropout-free survival    float64
dtype: object

In [45]:
# Clean and prepare the data

df = df.copy()

# Remove duplicate rows
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().copy()

# Clean admission status labels
df["Admission Status"] = (
    df["Admission Status"]
    .astype("string")
    .str.strip()
    .str.title()
)

# Convert application dates
df["Application_Date"] = pd.to_datetime(
    df["Application_Date"],
    errors="coerce"
)

# Convert year to numeric
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")

print("Dataset shape after cleaning:", df.shape)
print("\nAdmission status categories:")
print(df["Admission Status"].value_counts(dropna=False))

print("\nMissing values after cleaning:")
display(df.isnull().sum())

Duplicate rows: 0
Dataset shape after cleaning: (1000, 17)

Admission status categories:
Admission Status
Rejected    565
Accepted    435
Name: count, dtype: Int64

Missing values after cleaning:


Serial No.                 0
GRE Score                  0
TOEFL Score                0
University Rating          0
SOP                        0
LOR                        0
CGPA                       0
Research                   0
Chance of Admit            0
Year                       0
Application_Date         608
University_Name            0
region                     0
Admission Status           0
Duration_Months            0
Dropout_Event              0
Dropout-free survival      0
dtype: int64

In [46]:
# Calculate admission KPIs

total_applicants = len(df)

accepted_count = (
    df["Admission Status"].eq("Accepted").sum()
)

rejected_count = (
    df["Admission Status"].eq("Rejected").sum()
)

acceptance_rate = (
    accepted_count / total_applicants * 100
    if total_applicants > 0 else 0
)

print("Total applicants:", total_applicants)
print("Accepted applicants:", accepted_count)
print("Rejected applicants:", rejected_count)
print(f"Acceptance rate: {acceptance_rate:.2f}%")

year_summary = (
    df.dropna(subset=["Year"])
    .groupby("Year")["Admission Status"]
    .apply(lambda x: x.eq("Accepted").mean() * 100)
    .reset_index(name="Acceptance Rate (%)")
)

region_summary = (
    df.dropna(subset=["region"])
    .groupby("region")
    .agg(
        Total_Applicants=("Admission Status", "size"),
        Accepted=("Admission Status",
                  lambda x: x.eq("Accepted").sum())
    )
    .reset_index()
)

region_summary["Acceptance Rate (%)"] = (
    region_summary["Accepted"]
    / region_summary["Total_Applicants"] * 100
)

display(year_summary)
display(region_summary)

Total applicants: 1000
Accepted applicants: 435
Rejected applicants: 565
Acceptance rate: 43.50%


,Year,Acceptance Rate (%)
0,2022,46.0
1,2023,44.0
2,2024,40.5
3,2025,45.0
4,2026,42.0


,region,Total_Applicants,Accepted,Acceptance Rate (%)
0,Central,26,11,42.307692
1,East,115,44,38.260870
2,North,305,128,41.967213
3,Northeast,25,14,56.000000
4,South,379,159,41.952507
5,West,150,79,52.666667


In [47]:
# Create interactive visualizations

# A. Accepted vs. rejected applicants
status_counts = (
    df["Admission Status"]
    .value_counts()
    .rename_axis("Admission Status")
    .reset_index(name="Applicants")
)

fig = px.bar(
    status_counts,
    x="Admission Status",
    y="Applicants",
    color="Admission Status",
    title="Accepted vs. Rejected Applicants",
    text="Applicants"
)

fig.show()

In [48]:
# B. GRE score distribution.

fig = px.histogram(
    df,
    x="GRE Score",
    color="Admission Status",
    barmode="overlay",
    opacity=0.7,
    title="GRE Score Distribution by Admission Status"
)

fig.show()

In [49]:
# C. CGPA and admission status

fig = px.box(
    df,
    x="Admission Status",
    y="CGPA",
    color="Admission Status",
    title="CGPA Distribution by Admission Status"
)

fig.show()

In [50]:
# D. Admission trends over time
year_counts = (
    df.dropna(subset=["Year"])
    .groupby(["Year", "Admission Status"])
    .size()
    .reset_index(name="Applicants")
)

fig = px.line(
    year_counts,
    x="Year",
    y="Applicants",
    color="Admission Status",
    markers=True,
    title="Admission Outcomes by Year"
)

fig.show()

In [51]:
# E. Compare admission outcomes across regions

region_counts = (
    df.dropna(subset=["region"])
    .groupby(["region", "Admission Status"])
    .size()
    .reset_index(name="Applicants")
)

fig = px.bar(
    region_counts,
    x="region",
    y="Applicants",
    color="Admission Status",
    barmode="group",
    title="Admission Outcomes by Region"
)

fig.show()

In [52]:
# 6. Explore applicant records

numeric_columns = [
    "GRE Score",
    "TOEFL Score",
    "University Rating",
    "SOP",
    "LOR",
    "CGPA",
    "Research",
    "Chance of Admit"
]

display(df[numeric_columns].describe().round(2))

# compare average scores by admission outcom
score_summary = (
    df.groupby("Admission Status")[
        ["GRE Score", "TOEFL Score", "CGPA", "SOP", "LOR"]
    ]
    .mean()
    .round(2)
)

display(score_summary)

# Explore individual applicant records
selected_status = "Accepted"

applicant_view = df[
    df["Admission Status"] == selected_status
]

display(applicant_view.head(20))

,GRE Score,TOEFL Score,University Rating,SOP,LOR,CGPA,Research,Chance of Admit
count,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.0,1000.00
mean,315.96,106.56,3.02,3.39,3.41,8.59,0.5,0.72
std,13.42,7.63,1.32,1.03,0.89,0.61,0.5,0.15
min,290.00,92.00,1.00,0.90,0.90,6.77,0.0,0.32
25%,305.00,101.00,2.00,2.50,3.00,8.18,0.0,0.64
50%,316.00,107.00,3.00,3.50,3.50,8.58,1.0,0.72
75%,326.25,113.00,4.00,4.00,4.00,9.07,1.0,0.82
max,340.00,120.00,5.00,5.10,5.10,9.95,1.0,0.97


,GRE Score,TOEFL Score,CGPA,SOP,LOR
Admission Status,,,,,
Accepted,319.55,108.65,8.78,3.66,3.61
Rejected,313.20,104.96,8.44,3.19,3.25


,Serial No.,GRE Score,TOEFL Score,University Rating,SOP,LOR,CGPA,Research,Chance of Admit,Year,Application_Date,University_Name,region,Admission Status,Duration_Months,Dropout_Event,Dropout-free survival
3,147,315,105,3,2.0,2.5,8.48,0,0.75,2022,NaT,Indian Institute of Technology Delhi,North,Accepted,33,0,0.597989
6,150,311,106,2,3.5,3.0,8.26,1,0.79,2022,NaT,Indian Institute of Technology Madras,South,Accepted,35,0,0.560210
7,36,320,110,5,5.0,5.0,9.20,1,0.88,2022,NaT,Indian Institute of Technology Kanpur,North,Accepted,36,0,0.536473
8,45,326,113,5,4.5,4.0,9.40,1,0.91,2022,NaT,Indian Institute of Technology Kanpur,North,Accepted,41,1,0.473582
10,127,323,113,3,4.0,3.0,9.32,1,0.85,2022,NaT,Indian Institute of Technology Kanpur,North,Accepted,42,0,0.460896
11,196,307,107,2,3.0,3.5,8.52,1,0.78,2022,2022-01-02,Indian Institute of Technology Kharagpur,East,Accepted,35,0,0.560210
12,7,321,109,3,3.0,4.0,8.20,1,0.75,2022,2022-01-02,Indian Institute of Technology Kharagpur,East,Accepted,31,0,0.646808
13,1,337,118,4,4.5,4.5,9.65,1,0.92,2022,2022-02-02,Indian Institute of Technology Kharagpur,East,Accepted,48,0,0.389948
14,172,334,117,5,4.0,4.5,9.07,1,0.89,2022,2022-02-02,Indian Institute of Technology Kharagpur,East,Accepted,45,0,0.421283
16,33,338,118,4,3.0,4.5,9.40,1,0.91,2022,2022-03-02,Indian Institute of Technology Kharagpur,East,Accepted,44,0,0.430060


In [53]:
# Train and compare machine-learning models

# Prepare features and target
features = [
    "GRE Score",
    "TOEFL Score",
    "University Rating",
    "SOP",
    "LOR",
    "CGPA",
    "Research"
]

model_df = df.dropna(subset=["Admission Status"]).copy()

X = model_df[features].apply(pd.to_numeric, errors="coerce")
y = model_df["Admission Status"]

# Keep only rows with one of the two expected labels
valid_labels = y.isin(["Accepted", "Rejected"])
X = X.loc[valid_labels]
y = y.loc[valid_labels]

print("Features used:", features)
print("\nTarget distribution:")
print(y.value_counts())

Features used: ['GRE Score', 'TOEFL Score', 'University Rating', 'SOP', 'LOR', 'CGPA', 'Research']

Target distribution:
Admission Status
Rejected    565
Accepted    435
Name: count, dtype: Int64


In [54]:
# Split the data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 800
Testing rows: 200


In [55]:
# Train three models and compare them

models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000))
    ]),

    "Decision Tree": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", DecisionTreeClassifier(
            max_depth=5,
            random_state=42
        ))
    ]),

    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=200,
            max_depth=8,
            random_state=42,
            class_weight="balanced"
        ))
    ])
}

results = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    trained_models[name] = model

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(
            y_test, predictions,
            pos_label="Accepted",
            zero_division=0
        ),
        "Recall": recall_score(
            y_test, predictions,
            pos_label="Accepted",
            zero_division=0
        ),
        "F1 Score": f1_score(
            y_test, predictions,
            pos_label="Accepted",
            zero_division=0
        )
    })

results_df = pd.DataFrame(results).sort_values(
    by="F1 Score",
    ascending=False
)

display(results_df.round(3))

,Model,Accuracy,Precision,Recall,F1 Score
2,Random Forest,0.680,0.677,0.506,0.579
0,Logistic Regression,0.660,0.651,0.471,0.547
1,Decision Tree,0.635,0.625,0.402,0.490


In [56]:
# Plot model performance
metrics_long = results_df.melt(
    id_vars="Model",
    value_vars=["Accuracy", "Precision", "Recall", "F1 Score"],
    var_name="Metric",
    value_name="Score"
)

fig = px.bar(
    metrics_long,
    x="Model",
    y="Score",
    color="Metric",
    barmode="group",
    title="Model Performance Comparison",
    range_y=[0, 1]
)

fig.show()

In [57]:
# Inspect the best model and its classification report

best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]

print("Best model by test F1 score:", best_model_name)

best_predictions = best_model.predict(X_test)

print(
    classification_report(
        y_test,
        best_predictions,
        labels=["Accepted", "Rejected"],
        zero_division=0
    )
)

Best model by test F1 score: Random Forest
              precision    recall  f1-score   support

    Accepted       0.68      0.51      0.58        87
    Rejected       0.68      0.81      0.74       113

    accuracy                           0.68       200
   macro avg       0.68      0.66      0.66       200
weighted avg       0.68      0.68      0.67       200



In [58]:
# Plot a confusion matrix

cm = confusion_matrix(
    y_test,
    best_predictions,
    labels=["Accepted", "Rejected"]
)

fig = px.imshow(
    cm,
    x=["Predicted Accepted", "Predicted Rejected"],
    y=["Actual Accepted", "Actual Rejected"],
    text_auto=True,
    color_continuous_scale="Blues",
    title=f"Confusion Matrix — {best_model_name}"
)

fig.show()

In [59]:
# Save the result

results_df.to_csv("model_performance_results.csv", index=False)
df.to_csv("cleaned_admission_data.csv", index=False)

print("Analysis results saved successfully!")

Analysis results saved successfully!
